In [0]:
# %sql
# CREATE TABLE dp_750.default.bronze_orders AS
# SELECT *
# FROM read_files(
#     '/Volumes/dp_750/default/fulfulment-pulse/fictitious_orders_melbourne.csv',
#     format => 'csv',
#     header => true,
#     inferSchema => true
# );

In [0]:
# %sql

# CREATE TABLE dp_750.default.bronze_temperature AS 
# SELECT *
# FROM read_files(
#     '/Volumes/dp_750/default/fulfulment-pulse/Temperature_Data.csv',
#     format => 'csv',
#     header => true,
#     schema => 'Product_code STRING,
#     Bureau_of_Meteorology_station_number STRING,
#     Year INT,
#     Month INT,
#     Day INT,
#     Maximum_temperature_Degree_C FLOAT,
#     Days_of_accumulation_of_maximum_temperature FLOAT,
#     Quality STRING'
# );

In [0]:
# %sql
# CREATE TABLE dp_750.default.bronze_rainfall AS
# SELECT *
# FROM read_files(
#     '/Volumes/dp_750/default/fulfulment-pulse/rainfall_Data.csv',   
#     format => 'csv',
#     header => true,
#     schema => 'Product_code STRING,
#     Bureau_of_Meteorology_station_number STRING,
#     Year INT,
#     Month INT,
#     Day INT,
#     Rainfall_amount_millimetres FLOAT,
#     Days_of_accumulation_of_rainfall FLOAT,
#     Quality STRING');

 

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_orders
WHERE (order_id, order_timestamp, delivery_zone, item_count) IS NULL

In [0]:
%sql
SELECT order_id
FROM dp_750.default.bronze_orders
WHERE (item_count = 0) OR (item_count REGEXP '.');


In [0]:
%sql
SELECT order_id
FROM dp_750.default.bronze_orders
WHERE (item_count % 1 <> 0) OR (item_count <= 0)

In [0]:
%sql
SELECT order_id
FROM dp_750.default.bronze_orders
GROUP BY order_id
HAVING count(order_id) > 1


In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_orders 
WHERE is_synthetic = FALSE OR is_synthetic IS NULL

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_orders 
WHERE (delivery_zone IS NULL) OR TRIM(delivery_zone) = ""

In [0]:
%sql
SELECT order_timestamp
FROM dp_750.default.bronze_orders
WHERE order_timestamp < '2026-01-01T08:10:00.000+00:00'

In [0]:
%sql
SELECT order_timestamp
FROM dp_750.default.bronze_orders
WHERE order_timestamp > '2026-01-01T08:10:00.000+00:00'

In [0]:
%sql
CREATE TABLE dp_750.default.silver_orders AS
SELECT *
FROM dp_750.default.bronze_orders;

In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_temperature
WHERE Maximum_temperature_Degree_C IS NULL


In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_temperature

In [0]:
%sql
SELECT COUNT(Product_code), COUNT(Bureau_of_Meteorology_station_number), COUNT(Maximum_temperature_Degree_C), COUNT(Days_of_accumulation_of_maximum_temperature), 
COUNT(Quality),
COUNT(YEAR), 
COUNT(MONTH), 
COUNT(DAY)
FROM dp_750.default.bronze_temperature
GROUP BY Product_code, Bureau_of_Meteorology_station_number, Maximum_temperature_Degree_C, Days_of_accumulation_of_maximum_temperature
, Quality, YEAR, MONTH, DAY

HAVING COUNT(*) > 1
 

In [0]:
%sql
SELECT COUNT(Product_code)
FROM dp_750.default.bronze_temperature
GROUP BY Product_code


In [0]:
%sql
SELECT COUNT(Bureau_of_Meteorology_station_number)
FROM dp_750.default.bronze_temperature
GROUP BY Bureau_of_Meteorology_station_number


In [0]:
%sql
SELECT COUNT(DISTINCT(YEAR, MONTH, DAY))
FROM dp_750.default.bronze_temperature
GROUP BY YEAR, MONTH, DAY
HAVING COUNT(DISTINCT(YEAR, MONTH, DAY)) > 1

In [0]:
%sql
SELECT 
COUNT(*) - COUNT(product_code) AS NULLS_IN_PRODUCT_CODE,
COUNT(*) - COUNT(Bureau_of_Meteorology_station_number) AS NULLS_IN_Bureau_of_Meteorology_station_number, 
COUNT(*) - COUNT(Year) AS NULLS_IN_year,
COUNT(*) - COUNT(Month) AS NULLS_IN_month,
COUNT(*) - COUNT(day) AS NULLS_IN_day,
COUNT(*) - COUNT(Maximum_temperature_Degree_C) AS Maximum_temperature_Degree_C,
COUNT(*) - COUNT(Days_of_accumulation_of_maximum_temperature) AS NULLS_IN_Days_of_accumulation_of_maximum_temperature, 
COUNT(*) - COUNT(Quality) AS NULLS_IN_Q

FROM dp_750.default.bronze_temperature




In [0]:
%sql
SELECT *
FROM dp_750.default.bronze_temperature
GROUP BY product_code,
Bureau_of_Meteorology_station_number,
Year,
Month,
day,
Maximum_temperature_Degree_C,
Days_of_accumulation_of_maximum_temperature,
Quality
HAVING COUNT(*) > 1



In [0]:
%sql
SELECT 
COUNT(*) - COUNT(product_code) AS NULLS_IN_PRODUCT_CODE,
COUNT(*) - COUNT(Bureau_of_Meteorology_station_number) AS NULLS_IN_Bureau_of_Meteorology_station_number, 
COUNT(*) - COUNT(Year) AS NULLS_IN_year,
COUNT(*) - COUNT(Month) AS NULLS_IN_month,
COUNT(*) - COUNT(day) AS NULLS_IN_day,
COUNT(*) - COUNT(Maximum_temperature_Degree_C) AS Maximum_temperature_Degree_C,
COUNT(*) - COUNT(Days_of_accumulation_of_maximum_temperature) AS NULLS_IN_Days_of_accumulation_of_maximum_temperature, 
COUNT(*) - COUNT(Quality) AS NULLS_IN_Q

FROM dp_750.default.bronze_temperature


In [0]:
%sql
SELECT make_date(YEAR, MONTH, DAY) AS observation_date 
FROM dp_750.default.bronze_temperature



In [0]:
%sql
SELECT
    Bureau_of_Meteorology_station_number AS station_id,
    Year,
    Month,
    Day,
    Maximum_temperature_Degree_C AS temperature_c,
    Quality
FROM dp_750.default.bronze_temperature
WHERE Maximum_temperature_Degree_C < -10
   OR Maximum_temperature_Degree_C > 50
ORDER BY temperature_c;